In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
json2jsonl.py - 把 JSON 文件内容转成 JSONL（每行一个 JSON 对象）
直接修改下面【配置区】的 INPUT_FILE / OUTPUT_FILE 即可，无需传参。
用法: python json2jsonl.py
"""

import json
import ast
import sys

# ==================== 配置区（在这里改路径） ====================
INPUT_FILE = "/home/shaolingxuan/project/Search-o1/outputs/runs.baselines/seal0.qwen3.5-9b.search_o1/all-cleaned.json"    # 输入的 json 文件
OUTPUT_FILE = "/home/shaolingxuan/project/Search-o1/outputs/runs.baselines/seal0.qwen3.5-9b.search_o1/all-cleaned.jsonl"  # 输出的 jsonl 文件
ENSURE_ASCII = False  # False: 保留中文等原文; True: 转义成 \uXXXX
# ================================================================


def extract_objects(text: str):
    """从文本中提取所有顶层 JSON 对象，带容错。"""
    objs = []

    # ---- 尝试 1: 整体当标准 JSON 解析（数组或单对象）----
    try:
        data = json.loads(text)
        if isinstance(data, list):
            return data, "标准 JSON 数组"
        return [data], "单个 JSON 对象"
    except json.JSONDecodeError:
        pass

    # ---- 尝试 2: 逐行解析（NDJSON / 单引号 Python dict 行）----
    all_ok = True
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        obj = None
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            try:
                obj = ast.literal_eval(line)   # 兜底: {'a': 1} 单引号形式
            except Exception:
                all_ok = False
        if obj is not None:
            objs.append(obj)

    if all_ok and objs:
        return objs, "逐行 JSONL / Python dict"

    # ---- 尝试 3: 扫描大括号，逐个提取拼接的对象（如 {...} {...} 连在一起）----
    objs, depth, start, in_str, esc = [], 0, None, False, False
    for idx, ch in enumerate(text):
        if in_str:
            if esc:
                esc = False
            elif ch == '\\':
                esc = True
            elif ch == '"':
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch == '{':
            if depth == 0:
                start = idx
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0 and start is not None:
                chunk = text[start:idx + 1]
                try:
                    objs.append(json.loads(chunk))
                except json.JSONDecodeError:
                    try:
                        objs.append(ast.literal_eval(chunk))
                    except Exception as e:
                        print(f"警告: 无法解析片段 (位置 {idx}): {e}", file=sys.stderr)
                start = None
    return objs, "大括号扫描提取（多个对象拼接）"


def main():
    try:
        # 读文件，utf-8-sig 自动去掉 BOM
        with open(INPUT_FILE, "r", encoding="utf-8-sig") as f:
            text = f.read()
    except FileNotFoundError:
        sys.exit(f"❌ 错误: 找不到输入文件 {INPUT_FILE}")

    print(f"📖 读取文件: {INPUT_FILE}")
    objs, source_fmt = extract_objects(text)

    if not objs:
        sys.exit("❌ 错误: 未能从文件中提取出任何 JSON 对象")

    count = 0
    with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
        for i, obj in enumerate(objs, 1):
            try:
                out.write(json.dumps(obj, ensure_ascii=ENSURE_ASCII) + "\n")
                count += 1
            except (TypeError, ValueError) as e:
                print(f"⚠️  第 {i} 条对象序列化失败，已跳过: {e}", file=sys.stderr)

    print(f"✅ 转换完成: 共 {count} 条记录")
    print(f"   输入格式识别为: {source_fmt}")
    print(f"   输出文件: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


📖 读取文件: /home/shaolingxuan/project/Search-o1/outputs/runs.baselines/seal0.qwen3.5-9b.search_o1/all-cleaned.json
✅ 转换完成: 共 99 条记录
   输入格式识别为: 标准 JSON 数组
   输出文件: /home/shaolingxuan/project/Search-o1/outputs/runs.baselines/seal0.qwen3.5-9b.search_o1/all-cleaned.jsonl
